In [ ]:
# Transformer Forecasting - Version finale avec explications de dev

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1️⃣ Charger le dataset
print("\nÉtape 1 : Chargement du dataset")
df = pd.read_csv('scaled_dataset.csv')

print("Shape du dataframe:", df.shape)
print("Colonnes disponibles:", df.columns.tolist())

# 2️⃣ Préparation des colonnes
# Target = production | Features = tout le reste (sauf éventuellement 'date')
target_col = 'production'
exclude_cols = ['date', target_col] if 'date' in df.columns else [target_col]
features_cols = [col for col in df.columns if col not in exclude_cols]

print("\nFeatures utilisées:", features_cols)

# 3️⃣ Extraction des features et target
X_scaled = df[features_cols].values
y_scaled = df[[target_col]].values

print("\nShape X_scaled:", X_scaled.shape)
print("Shape y_scaled:", y_scaled.shape)

# 4️⃣ Création des séquences temporelles (fenêtre glissante)
def create_sequences(X, y, seq_len_input=30, seq_len_output=7):
    X_seqs, y_seqs = [], []
    for i in range(len(X) - seq_len_input - seq_len_output):
        X_seqs.append(X[i:i+seq_len_input])
        y_seqs.append(y[i+seq_len_input : i+seq_len_input+seq_len_output].flatten())
    return np.array(X_seqs), np.array(y_seqs)

print("\nÉtape 4 : Création des séquences temporelles")
seq_len_input = 30
seq_len_output = 7

X_seq, y_seq = create_sequences(X_scaled, y_scaled, seq_len_input, seq_len_output)

print("\nShape X_seq:", X_seq.shape)
print("Shape y_seq:", y_seq.shape)

# 5️⃣ Split en train / test
print("\nÉtape 5 : Split train/test")
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, shuffle=False
)

print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)

# 6️⃣ Positional Encoding
# Permet au Transformer de "savoir" dans quelle position se trouvent les tokens dans la séquence
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

# 7️⃣ Modèle Transformer Forecasting
# Architecture simple : Transformer Encoder + MLP decoder
class TransformerForecasting(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, seq_len_input, seq_len_output):
        super(TransformerForecasting, self).__init__()

        # Projette les features d'entrée vers d_model
        self.input_proj = nn.Linear(input_dim, d_model)

        # Positional encoding sur la séquence
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=seq_len_input)

        # Transformer Encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=512,
            dropout=0.1, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # MLP pour régresser sur les 7 jours de production future
        self.decoder = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, seq_len_output)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x_mean = x.mean(dim=1)
        y_pred = self.decoder(x_mean)
        return y_pred

# 8️⃣ Entraînement avec Scheduler LR et EarlyStopping
def train_model(model, X_train, y_train, X_test, y_test, n_epochs=100, batch_size=64, lr=1e-3, patience=10):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    test_losses = []
    r2_scores = []
    best_loss = np.inf
    patience_counter = 0

    for epoch in range(n_epochs):
        model.train()

        permutation = np.random.permutation(len(X_train))
        X_train_shuffled = X_train[permutation]
        y_train_shuffled = y_train[permutation]

        for i in range(0, len(X_train), batch_size):
            X_batch = torch.tensor(X_train_shuffled[i:i+batch_size], dtype=torch.float32).to(device)
            y_batch = torch.tensor(y_train_shuffled[i:i+batch_size], dtype=torch.float32).to(device)

            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

        # Évaluation test
        model.eval()
        with torch.no_grad():
            X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
            y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

            y_pred_test = model(X_test_tensor).cpu().numpy()
            y_true_test = y_test_tensor.cpu().numpy()

            test_loss = np.mean((y_pred_test - y_true_test) ** 2)
            r2 = r2_score(y_true_test.flatten(), y_pred_test.flatten())

        scheduler.step(test_loss)

        print(f"Epoch {epoch+1}/{n_epochs}, Test Loss: {test_loss:.4f}, R^2: {r2:.4f}")
        test_losses.append(test_loss)
        r2_scores.append(r2)

        # EarlyStopping
        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model_transformer.pth')
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"EarlyStopping déclenché à l'epoch {epoch+1}")
            break

    return model, test_losses, r2_scores

# 9️⃣ Instanciation et run du modèle
print("\nÉtape 9 : Lancement de l'entraînement du Transformer")
input_dim = X_train.shape[2]
d_model = 128
nhead = 8
num_layers = 4

model = TransformerForecasting(
    input_dim=input_dim,
    d_model=d_model,
    nhead=nhead,
    num_layers=num_layers,
    seq_len_input=seq_len_input,
    seq_len_output=seq_len_output
)

model, test_losses, r2_scores = train_model(
    model, X_train, y_train, X_test, y_test,
    n_epochs=100, batch_size=64, lr=1e-3, patience=10
)

# 🔟 Plot courbes Loss & R²
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(test_losses)
plt.title("Test Loss par Epoch")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid()

plt.subplot(1,2,2)
plt.plot(r2_scores)
plt.title("R² par Epoch")
plt.xlabel("Epoch")
plt.ylabel("R²")
plt.grid()

plt.tight_layout()
plt.show()

# ✅ Fin du training - modèle sauvegardé sous 'best_model_transformer.pth'
